# Test Remote vLLM Server

Kiểm tra kết nối tới vLLM trên ThunderCompute VM.

In [14]:
import requests, time, json, sys
from pathlib import Path

VLLM_BASE_URL = 'http://216.81.200.237:8000/v1'
MODEL_NAME    = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
TIMEOUT       = 30
COMPLETION_URL = f'{VLLM_BASE_URL}/chat/completions'

print(f'endpoint : {VLLM_BASE_URL}')
print(f'model    : {MODEL_NAME}')

endpoint : http://216.81.200.237:8000/v1
model    : Qwen/Qwen2.5-7B-Instruct-AWQ


## 1. Health Check

In [15]:
try:
    resp = requests.get(f'{VLLM_BASE_URL}/models', timeout=TIMEOUT)
    if resp.status_code == 200:
        print('✓ vLLM ready')
        for m in resp.json().get('data', []):
            print(f'  model: {m["id"]}')
    else:
        print(f'✗ HTTP {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    print(f'✗ Cannot connect: {e}')

✗ Cannot connect: HTTPConnectionPool(host='216.81.200.237', port=8000): Max retries exceeded with url: /v1/models (Caused by NewConnectionError("HTTPConnection(host='216.81.200.237', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))


## 2. Simple Completion

In [ ]:
start = time.time()
try:
    resp = requests.post(COMPLETION_URL, json={
        'model': MODEL_NAME,
        'messages': [{'role': 'user', 'content': 'Say hello in one sentence.'}],
        'temperature': 0.0,
        'max_tokens': 50,
    }, timeout=TIMEOUT)
    elapsed = time.time() - start
    if resp.status_code == 200:
        print(f'✓ {elapsed:.1f}s | {resp.json()["choices"][0]["message"]["content"]}')
    else:
        print(f'✗ HTTP {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    print(f'✗ {e}')

## 3. JSON Mode (bắt buộc cho Type 1 pipeline)

In [ ]:
start = time.time()
try:
    resp = requests.post(COMPLETION_URL, json={
        'model': MODEL_NAME,
        'messages': [{'role': 'user', 'content': 'Return JSON: {"result": "ok"}'}],
        'temperature': 0.0,
        'max_tokens': 50,
        'response_format': {'type': 'json_object'},
    }, timeout=TIMEOUT)
    elapsed = time.time() - start
    if resp.status_code == 200:
        text = resp.json()['choices'][0]['message']['content']
        parsed = json.loads(text)
        print(f'✓ {elapsed:.1f}s | valid JSON: {parsed}')
    else:
        print(f'✗ HTTP {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    print(f'✗ {e}')

## 4. Benchmark Latency (5 requests)

In [ ]:
times = []
for i in range(5):
    start = time.time()
    try:
        resp = requests.post(COMPLETION_URL, json={
            'model': MODEL_NAME,
            'messages': [{'role': 'user', 'content': f'Q{i+1}: What is {i+1}+{i+1}?'}],
            'temperature': 0.0, 'max_tokens': 20,
        }, timeout=TIMEOUT)
        elapsed = time.time() - start
        times.append(elapsed)
        print(f'  [{i+1}/5] {elapsed:.2f}s {"✓" if resp.status_code == 200 else "✗"}')
    except Exception as e:
        print(f'  [{i+1}/5] ✗ {e}')

if times:
    print(f'\nMin={min(times):.2f}s | Max={max(times):.2f}s | Avg={sum(times)/len(times):.2f}s')

## 5. Integration Test — Exact2026 Type 1 Pipeline

In [ ]:
src_dir = Path().resolve().parent / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
print(f'src: {src_dir}')

In [ ]:
from exact.config import Settings
from exact.llm_client import build_json_client_from_settings
from exact.logic.translation.llm_translator import _build_premises_only_messages

client = build_json_client_from_settings(Settings().model_copy(update={
    'llm_provider': 'openai',
    'llm_base_url': VLLM_BASE_URL,
    'llm_model': MODEL_NAME,
    'llm_api_key': 'EMPTY',
    'llm_max_tokens': 2048,
    'llm_temperature': 0.0,
}))
print(f'✓ client: {type(client).__name__}')

premises = [
    'If a student attends at least 80% of classes, they are allowed to take the exam.',
    'If a student passes the exam, they pass the course.',
    'If a student fails the exam, they must retake the course.',
]
messages = _build_premises_only_messages(premises)
print(f'Translating {len(premises)} premises...')

start = time.time()
try:
    result = client.complete_json_sync(messages=messages, temperature=0.0, max_tokens=2048)
    elapsed = time.time() - start
    print(f'✓ {elapsed:.1f}s | predicates={len(result.get("predicates",[]))} premises={len(result.get("premises",[""]))}') 
except Exception as e:
    print(f'✗ {e}')